## **TDA Mapper**
### Segmentación de Mujeres Embarazadas según Exposición a Ácido Fólico y Riesgo Neonatal

### Resumen del pipeline

1. **Ingeniería de features**: cast numérico de columnas crudas; construcción de `prematuro`, `bajo_peso`, `riesgo_obstetrico`, `score_pan` y `AF_exposure`.
2. **Espacio TDA**: 10 variables normalizadas con `StandardScaler` tras imputación mediana; matriz resultante `X_scaled`.
3. **Grafo Mapper**: lente PCA 2D → cubierta `(n_cubes=15, overlap=50 %)` → `AgglomerativeClustering(k=4, ward)`.
4. **Componentes conexas**: extracción por Union-Find, filtro `n ≥ 20`, exportación a `mapper_clusters.csv` y `mapper_graph.pkl`.

In [23]:
import warnings
from pathlib import Path

import kmapper as km
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

data_path = Path('..') / 'Data' / 'Ingesta_AF_clean.csv'
print(data_path.resolve())
df = pd.read_csv(data_path)

print("Shape del dataset:", df.shape)
print("\nPrimeras filas:")
df.head()

C:\Users\valer\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Reto-Topologia\Data\Ingesta_AF_clean.csv
Shape del dataset: (1170, 73)

Primeras filas:


,Cod,edad,region_chile,neduc,paridad,fnacimiento,Sexo,pnacer,Edad_gestacion ultimo hijo,sitgest,...,pnacer_raw,eg_raw,uf_af,n_panes,pnacer_num,eg_num,n_panes_num,prematuro,eg_cat,uso_supl
0,1,26.0,1,4,1,2015-01-15 00:00:00,2,3040,39,9,...,3040.0,39.0,1000.0,2.0,3040.0,39.0,2.0,0.0,39–40 (normal),Usó suplemento
1,2,36.0,1,6,2,2012-03-29 00:00:00,2,3875,39,5,...,3875.0,39.0,1000.0,3.0,3875.0,39.0,3.0,0.0,39–40 (normal),Usó suplemento
2,3,22.0,1,5,1,2011-11-22 00:00:00,1,2665,38,0,...,2665.0,38.0,NaN,2.0,2665.0,38.0,2.0,0.0,37–38 (temprano),No usó suplemento
3,4,35.0,1,3,2,2007-02-14 00:00:00,1,3450,37,0,...,3450.0,37.0,1000.0,2.0,3450.0,37.0,2.0,0.0,37–38 (temprano),Usó suplemento
4,5,33.0,1,5,0,NaN,0,NaN,NaN,0,...,NaN,NaN,NaN,3.0,NaN,NaN,3.0,NaN,NaN,No usó suplemento


In [24]:
selected_cols = [
    "uf-af",
    "N° PANES",
    "Marraqueta",
    "Hallulla",
    "Pan molde integral",
    "Antes del embarazo",
    "Durante todo el embarazo",
    "1-3 meses",
    "4-6 meses",
    "7-9 meses",
    "NO consumio",
    "pnacer",
    "Edad_gestacion ultimo hijo",
    "Preeclamsia",
    "Diabetes gestacional",
    "Embarazo multiple",
    "edad",
    "neduc",
]

data = df[selected_cols].copy()

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data["prematuro"] = (data["Edad_gestacion ultimo hijo"] < 37).astype(int)  # < 37 semanas
data["bajo_peso"]  = (data["pnacer"] < 2500).astype(int)                  # < 2500 g

data["riesgo_obstetrico"] = (
    data["Preeclamsia"].fillna(0)
    + data["Diabetes gestacional"].fillna(0)
    + data["Embarazo multiple"].fillna(0)
    + data["bajo_peso"].fillna(0)
    + data["prematuro"].fillna(0)
)

# 160 µg/pan: estándar de fortificación obligatoria en Chile (DS 977/96)
data["score_pan"]   = data["N° PANES"].fillna(0) * 160
data["AF_exposure"] = data["uf-af"].fillna(0) + data["score_pan"].fillna(0)

print("\nData con features derivadas:")
data.head()


Data con features derivadas:


,uf-af,N° PANES,Marraqueta,Hallulla,Pan molde integral,Antes del embarazo,Durante todo el embarazo,1-3 meses,4-6 meses,7-9 meses,...,Preeclamsia,Diabetes gestacional,Embarazo multiple,edad,neduc,prematuro,bajo_peso,riesgo_obstetrico,score_pan,AF_exposure
0,1000.0,2.0,0,1,0,0,0,0,0,0,...,0,0.0,0.0,26.0,4,0,0,0.0,320.0,1320.0
1,1000.0,3.0,1,0,0,0,0,1,0,0,...,0,1.0,0.0,36.0,6,0,0,1.0,480.0,1480.0
2,NaN,2.0,1,0,0,0,0,0,0,0,...,0,0.0,0.0,22.0,5,0,0,0.0,320.0,320.0
3,1000.0,2.0,1,0,0,0,0,1,0,0,...,0,0.0,0.0,35.0,3,0,0,0.0,320.0,1320.0
4,NaN,3.0,1,0,0,0,0,0,0,0,...,0,0.0,0.0,33.0,5,0,0,0.0,480.0,480.0


In [25]:
data["riesgo_obstetrico"].describe()

count    1170.000000
mean        0.280342
std         0.587908
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         3.000000
Name: riesgo_obstetrico, dtype: float64

### Parámetros del grafo Mapper

El algoritmo Mapper resume la topología del dataset en tres pasos:

1. **Función filtro**: proyección PCA 2D sobre `X_scaled` — reduce dimensión preservando la métrica global.
2. **Cubierta**: `km.Cover(n_cubes=15, perc_overlap=0.50)` — divide el espacio de la lente en regiones superpuestas; el solapamiento garantiza continuidad entre nodos adyacentes.
3. **Clustering local**: `AgglomerativeClustering(n_clusters=4, linkage='ward')` — cada clúster dentro de una región se convierte en un nodo; dos nodos se unen si comparten observaciones.

La variable de colorización es **`AF_exposure`** (suplementación directa + fortificación por pan), lo que permite identificar subpoblaciones por nivel de ingesta de ácido fólico.

In [26]:
tda_features = [
    # Exposición AF
    "AF_exposure",
    "uf-af",
    "score_pan",
    # Timing de suplementación
    "Antes del embarazo",
    "Durante todo el embarazo",
    "1-3 meses",
    "4-6 meses",
    "7-9 meses",
    # Características maternas
    "edad",
    "neduc",
]

X = data[tda_features].copy()

In [27]:
imputer   = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Forma de X_scaled:", X_scaled.shape)

Forma de X_scaled: (1170, 10)


In [28]:
mapper = km.KeplerMapper(verbose=1)

lens = mapper.project(
    X_scaled,
    projection=PCA(n_components=2, random_state=42),
)

KeplerMapper(verbose=1)
..Projecting on data shaped (1170, 10)

..Projecting data using: 
	PCA(n_components=2, random_state=42)


..Scaling with: MinMaxScaler()



In [29]:
cover = km.Cover(
    n_cubes=15,         # intervalos por dimensión
    perc_overlap=0.50,  # solapamiento entre cubos adyacentes
)

In [30]:
clusterer = AgglomerativeClustering(
    n_clusters=4,
    linkage='ward',
)

In [31]:
graph = mapper.map(
    lens,
    X_scaled,
    cover=cover,
    clusterer=clusterer,
)

print("Nodos:", len(graph["nodes"]))
print("Links:", len(graph["links"]))

Mapping on data shaped (1170, 10) using lens shaped (1170, 2)

Creating 225 hypercubes.

Created 1507 edges and 564 nodes in 0:00:00.353350.
Nodos: 564
Links: 491


In [32]:
# OPCIÓN A — exposición total a ácido fólico
color_values = X_imputed["AF_exposure"]
color_name   = "Exposición Total AF"

# OPCIÓN B — prematuridad
# color_values = X_imputed["prematuro"]
# color_name   = "Prematuridad"

# OPCIÓN C — peso al nacer
# color_values = X_imputed["pnacer"]
# color_name   = "Peso al nacer"

html = mapper.visualize(
    graph,
    path_html="mapper_af_embarazo_semifinal.html",
    title="""
    TDA Mapper — Segmentación de Mujeres Embarazadas
    según Exposición a Ácido Fólico y Riesgo Neonatal
    """,
    color_values=color_values,
    color_function_name=color_name,
)

Wrote visualization to: mapper_af_embarazo_semifinal.html


In [33]:
from collections import defaultdict
from pathlib import Path
import pickle

print('=' * 70)
print('COMPONENTES CONECTADOS — GRAFO MAPPER')
print('=' * 70)

print(f'\nEstructura de graph["links"]:')
print(f'  Tipo: {type(graph["links"])}')
print(f'  Número de elementos: {len(graph["links"])}')
if len(graph['links']) > 0:
    print(f'  Primer elemento: {graph["links"][0]}')
    print(f'  Tipo del primer elemento: {type(graph["links"][0])}')

parent: dict = {}


def find(x: str) -> str:
    """Encuentra el representante de la componente con path compression."""
    if x not in parent:
        parent[x] = x
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]


def union(x: str, y: str) -> None:
    """Une las componentes de x e y."""
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py


for node_id in graph['nodes'].keys():
    find(node_id)

print(f'\nProcesando links...')
for link_item in graph['links']:
    if isinstance(link_item, (list, tuple)) and len(link_item) == 2:
        # Formato: [node_a, node_b] o (node_a, node_b)
        node_a, node_b = link_item
    elif isinstance(link_item, dict):
        # Formato: {"source": node_a, "target": node_b} o similar
        if 'source' in link_item and 'target' in link_item:
            node_a, node_b = link_item['source'], link_item['target']
        else:
            vals = list(link_item.values())
            if len(vals) >= 2:
                node_a, node_b = vals[0], vals[1]
            else:
                print(f'  ⚠ Formato de link no reconocido: {link_item}')
                continue
    else:
        print(f'  ⚠ Formato de link no reconocido: {link_item} ({type(link_item)})')
        continue
    union(node_a, node_b)

components = defaultdict(list)
for node_id in graph['nodes'].keys():
    comp_id = find(node_id)
    components[comp_id].append(node_id)

print(f'\nComponentes conectados encontrados: {len(components)}')
for i, (comp_id, nodes) in enumerate(sorted(components.items())):
    n_obs = sum(len(graph['nodes'][n]) for n in nodes)
    print(f'  Componente {i}: {len(nodes)} nodos \u2192 {n_obs} observaciones')

point_to_component = {}
for comp_id, nodes_in_comp in components.items():
    for node_id in nodes_in_comp:
        for point_idx in graph['nodes'][node_id]:
            try:
                point_idx_int = int(point_idx)
            except (ValueError, TypeError):
                point_idx_int = point_idx
            point_to_component[point_idx_int] = comp_id

assign = pd.Series(index=data.index, dtype="object")
for pos, comp_id in point_to_component.items():
    if pos < len(data):
        orig_idx = data.index[pos]
        assign.loc[orig_idx] = comp_id

data["cluster_mapper"] = assign

comp_counts = data['cluster_mapper'].value_counts()
valid_comps = comp_counts[comp_counts >= 20].index.tolist()

print(f'\nFiltro: componentes con n >= 20')
print(f'  Componentes válidos: {len(valid_comps)} de {len(components)}')
for comp in sorted(valid_comps):
    n = comp_counts[comp]
    print(f'    Componente {comp}: n={int(n)}')

comp_remapping = {old_comp: new_id for new_id, old_comp in enumerate(sorted(valid_comps))}
data["cluster_mapper"] = data["cluster_mapper"].map(comp_remapping)

print(f'\nResumen final:')
print(f'  Observaciones con componente válido: {data["cluster_mapper"].notna().sum()} / {len(data)}')
print(f'  Componentes en análisis: {data["cluster_mapper"].max() + 1 if data["cluster_mapper"].notna().any() else 0}')

graph_path = Path('mapper_graph.pkl')
try:
    with open(graph_path, 'wb') as f:
        pickle.dump(graph, f)
    print(f'\n\u2713 Grafo guardado en {graph_path}')
except Exception as e:
    print(f'\n\u2717 Error guardando grafo: {e}')

clusters_df = pd.DataFrame({
    "orig_index":     data.index,
    "cluster_mapper": data["cluster_mapper"].values,
})

save_paths = [Path('Data') / 'mapper_clusters.csv', Path('mapper_clusters.csv')]
for p in save_paths:
    try:
        p.parent.mkdir(parents=True, exist_ok=True)
        clusters_df.to_csv(p, index=False)
        print(f'\u2713 CSV guardado: {p}')
    except Exception as e:
        print(f'\u2717 Error guardando CSV en {p}: {e}')

print('=' * 70)

COMPONENTES CONECTADOS — GRAFO MAPPER

Estructura de graph["links"]:
  Tipo: <class 'collections.defaultdict'>
  Número de elementos: 491
  Primer elemento: []
  Tipo del primer elemento: <class 'list'>

Procesando links...
  ⚠ Formato de link no reconocido: cube1_cluster0 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube1_cluster1 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube1_cluster2 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube1_cluster3 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube2_cluster0 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube2_cluster1 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube2_cluster2 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube2_cluster3 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube3_cluster0 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube3_cluster1 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube3_cluster2 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube3_cluster3 

In [ ]:

# Exportar datos completos con asignación de cluster
export_df = data.copy()
export_df.insert(0, "orig_index", data.index)

output_path = Path("mapper_resultados_completos.csv")
export_df.to_csv(output_path, index=False)
print(f"✓ Resultados exportados: {output_path.resolve()}")
print(f"  Shape: {export_df.shape}")
print(f"  Columnas: {list(export_df.columns)}")
print(f"\nDistribución de clusters:")
print(export_df["cluster_mapper"].value_counts(dropna=False).sort_index())
